# PySpark 2: DataFrame essentials

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/pyspark/pyspark-02-dataframe-essentials.ipynb)

An analyst asks for "paid orders over 100, newest first, with the customer name and a tax column". Every
PySpark pipeline is built from a few dozen operations like these: pick columns, filter rows, add columns,
sort, combine. This lesson teaches them one at a time.

You will create DataFrames, inspect their schema and contents, select and compute columns, filter, sort and
limit rows, remove duplicates, combine DataFrames, take samples, and bring results back to Python safely.

Each section explains how something works, where it matters and what to watch out for, then runs small examples. Read each output, then the **What to notice** note under it. Run the cells in order.

**Prerequisites:** [PySpark 1: how Spark runs your code](pyspark-01-how-spark-runs.ipynb).

*Last reviewed: November 2026, against Apache Spark 4.0.*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0

## Setup and sample data

The first cell starts a local Spark session. The second creates `customers` (one row per customer; customer 4 has no region, and 7 and 8 have never ordered), `orders` (one row per order; order 12 lands just after midnight UTC) and `order_items` (one row per order line; each order's lines add up to its amount): a slice of the orders platform that every
module shares, introduced in [PySpark 1](pyspark-01-how-spark-runs.ipynb#sample-data-used-in-every-module).

In [ ]:
import os
import sys
import tempfile
import time

from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable  # Python workers use this notebook's interpreter
os.environ["TZ"] = "UTC"  # the sample timestamps are UTC
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-")  # scratch space for the files the lessons write

spark = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")  # the default of 200 is far too many for tiny data
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
from datetime import date, datetime
from decimal import Decimal as D

from pyspark.sql import functions as F

customers = spark.createDataFrame([
    (1, "Acme Corp", "North", date(2024, 1, 3)),
    (2, "Bright Retail", "North", date(2024, 2, 11)),
    (3, "Cyan Traders", "South", date(2024, 3, 15)),
    (4, "Delta Grocers", None, date(2024, 5, 2)),  # region never arrived from the CRM
    (5, "Echo Supplies", "West", date(2024, 6, 20)),
    (6, "Foxtrot Foods", "West", date(2024, 8, 1)),
    (7, "Gamma Sports", "East", date(2024, 9, 14)),  # 7 and 8 have never ordered
    (8, "Hotel Homeware", "East", date(2024, 11, 5)),
], "customer_id int, name string, region string, signup_date date")

orders = spark.createDataFrame([
    (1, 1, "paid", datetime(2026, 1, 5, 9, 15), D("120.00")),
    (2, 1, "shipped", datetime(2026, 1, 5, 10, 5), D("84.50")),
    (3, 1, "returned", datetime(2026, 1, 5, 11, 40), D("30.00")),
    (4, 2, "paid", datetime(2026, 1, 6, 15, 20), D("260.00")),
    (5, 2, "placed", datetime(2026, 1, 7, 8, 5), D("55.25")),
    (6, 3, "shipped", datetime(2026, 1, 7, 17, 45), D("149.00")),
    (7, 3, "paid", datetime(2026, 1, 9, 12, 10), D("310.00")),
    (8, 4, "returned", datetime(2026, 1, 10, 19, 30), D("89.90")),
    (9, 4, "shipped", datetime(2026, 1, 12, 8, 55), D("1180.00")),
    (10, 5, "paid", datetime(2026, 1, 12, 13, 0), D("74.00")),
    (11, 6, "placed", datetime(2026, 1, 13, 10, 25), D("210.00")),
    (12, 6, "shipped", datetime(2026, 1, 14, 0, 30), D("96.40")),  # just after midnight UTC
], "order_id int, customer_id int, status string, order_ts timestamp, amount decimal(12,2)")

order_items = spark.createDataFrame([
    (1, "SKU-4001", 2, D("12.50")), (1, "SKU-4007", 1, D("95.00")),
    (2, "SKU-4002", 1, D("34.50")), (2, "SKU-4009", 2, D("25.00")),
    (3, "SKU-4002", 1, D("30.00")),
    (4, "SKU-4003", 1, D("149.00")), (4, "SKU-4004", 2, D("55.50")),
    (5, "SKU-4005", 1, D("55.25")),
    (6, "SKU-4003", 1, D("149.00")),
    (7, "SKU-4006", 1, D("189.00")), (7, "SKU-4008", 1, D("121.00")),
    (8, "SKU-4010", 1, D("89.90")),
    (9, "SKU-4011", 20, D("59.00")),  # a bulk reorder
    (10, "SKU-4012", 2, D("37.00")),
    (11, "SKU-4013", 1, D("210.00")),
    (12, "SKU-4014", 1, D("41.40")), (12, "SKU-4005", 1, D("55.00")),
], "order_id int, sku string, qty int, unit_price decimal(8,2)")

## Create DataFrames

A DataFrame is a distributed table with a **schema**: column names and types. In tests and small examples you
create one from Python data with `spark.createDataFrame(rows, schema)`. The schema can be:

- a **DDL string**, such as `"order_id int, status string"`: short and readable;
- a `StructType` of `StructField`s: verbose, but it lets you set nullability and build schemas in code;
- omitted, in which case Spark infers types from the data (avoid this outside quick experiments).

`spark.range(n)` creates a single `id` column of `0..n-1`, handy for generating test data. In real pipelines
DataFrames come from files and tables ([lesson 4](pyspark-04-reading-and-writing-data.ipynb)).

**Where it matters**
- **Unit tests:** a few hand-written rows with an explicit schema.
- **Lookup tables** small enough to write inline, such as status codes.

**Watch out for**
- Inferred schemas guess: integers become `bigint`, decimals become `double`, and a column of only `None` fails.
- Rows as tuples must match the schema's column order; rows as dicts or `Row`s match by name.

### Example

Create the same small table three ways and compare the schemas.

In [ ]:
from pyspark.sql import Row
from pyspark.sql import types as T

ddl = spark.createDataFrame([("o1", 120.5), ("o2", None)], "order_id string, amount double")

struct = spark.createDataFrame(
    [("o1", 120.5), ("o2", None)],
    T.StructType([
        T.StructField("order_id", T.StringType(), nullable=False),
        T.StructField("amount", T.DoubleType(), nullable=True),
    ]),
)

inferred = spark.createDataFrame([Row(order_id="o1", amount=120.5), Row(order_id="o2", amount=None)])

for name, df in [("DDL string", ddl), ("StructType", struct), ("inferred", inferred)]:
    print(f"{name:<11}", df.schema.simpleString(), "| order_id nullable:", df.schema["order_id"].nullable)
spark.range(3).show()

**What to notice:** All three have the same column types here because the Python values were unambiguous. Only the `StructType`
version records that `order_id` cannot be null. `spark.range(3)` produced one `bigint` column named `id`.

## Inspect a DataFrame

Before transforming data, look at it:

- `printSchema()` shows names, types and nullability as a tree; `columns` and `dtypes` return them as lists.
- `show(n, truncate=False)` prints rows; `show(vertical=True)` prints one field per line, useful for wide rows.
- `count()` counts rows (an action that reads all the data).
- `describe()` and `summary()` compute count, mean, standard deviation, min and max (and, for `summary`,
  percentiles) for every column.

**Where it matters**
- **First contact with a new source:** types, nulls and value ranges before you write any logic.
- **Debugging:** `printSchema()` after a join or a read shows unexpected column names or types at once.

**Watch out for**
- `show()` prints 20 rows and truncates strings at 20 characters by default.
- `describe()` and `summary()` scan the whole dataset; on a large table, run them on a sample.

### Example

Inspect `orders`: schema, types, a vertical view of one row, and summary statistics for the amount.

In [ ]:
orders.printSchema()
print(orders.dtypes)
orders.show(1, vertical=True)
orders.select("amount").summary("count", "min", "25%", "50%", "max").show()

**What to notice:** `amount` is a `decimal(12,2)`, so sums stay exact. The vertical view shows every column of the first row on its
own line. The median order (`50%`) is 96.4 while the maximum is 1,180.00: one bulk order is far larger than the
rest, which matters for any average you report. `summary` computes percentiles approximately and prints them as
doubles, which is why they lose the trailing zeros.

## Select and compute columns

`select` returns a new DataFrame with the columns you list. A column can be referenced as a string (`"amount"`),
with `F.col("amount")`, or as `orders["amount"]` (useful when two DataFrames share column names). Combine
columns with arithmetic and functions to make new ones, and name them with `alias`.

`selectExpr` and `F.expr` accept SQL expressions as strings, which is often shorter for arithmetic and `CASE`.
`F.lit(value)` turns a Python value into a column.

**Where it matters**
- **Shaping output:** keep only the columns a table promises its consumers, in a fixed order.
- **Mixing styles:** a SQL expression inside DataFrame code for one tricky calculation.

**Watch out for**
- A Python string in `select` is a column name, not a value. Wrap constants in `F.lit`.
- Computed columns without `alias` get generated names such as `(amount * 1.2)`.
- `F.lit(0.2)` is a `double`. Mixing it into decimal money turns the result into a floating-point number.

### Example

Select columns three ways, compute a tax column, and write the same projection with `selectExpr`.

In [ ]:
orders.select(
    "order_id",
    F.col("status"),
    orders["amount"],
    (F.col("amount") * F.lit(0.2)).alias("tax"),
    F.lit("EUR").alias("currency"),
).show(3)

orders.selectExpr("order_id", "amount * 0.2 AS tax", "upper(status) AS status").show(3)

**What to notice:** Look at the tax for order 2. `F.lit(0.2)` wraps a Python float, so Spark made the tax a `double`, and binary
floating point cannot store 16.9 exactly: `16.900000000000002`. In `selectExpr`, the SQL literal `0.2` is a
decimal, so the same calculation stays exact (`16.900`). For money, use `F.lit(Decimal("0.2"))` or SQL
literals. `F.lit("EUR")` added the same constant to every row.

## Add, replace, rename and drop columns

- `withColumn(name, expr)` adds a column, or replaces it if the name exists.
- `withColumns({name: expr, ...})` adds several in one step.
- `withColumnRenamed(old, new)` and `withColumnsRenamed({old: new})` rename.
- `drop(*names)` removes columns; dropping a name that does not exist is not an error.

**Where it matters**
- **Derived fields:** order date from a timestamp, a tier from an amount.
- **Contract fixes:** renaming a vendor's `ordrId` to your `order_id`.

**Watch out for**
- Calling `withColumn` hundreds of times in a loop builds a very deep plan that is slow to analyze. Use one `withColumns` or `select`.
- `withColumn` with an existing name silently replaces the column; check that this is what you want.

### Example

Add two columns at once, replace one, rename one and drop one.

In [ ]:
reshaped = (
    orders
    .withColumns({
        "order_date": F.to_date("order_ts"),
        "is_large": F.col("amount") >= 200,
    })
    .withColumn("status", F.upper("status"))  # replaces the existing column
    .withColumnRenamed("amount", "amount_eur")
    .drop("order_ts", "column_that_does_not_exist")
)
reshaped.printSchema()
reshaped.show(4)

**What to notice:** `status` kept its position but now holds upper-case values, `amount` became `amount_eur`, and `order_ts` is
gone. Dropping a column that does not exist raised no error, which is convenient but can hide a typo.

## Filter rows

`where` and `filter` are the same method. A condition is a boolean column built with comparison operators and
combined with `&` (and), `|` (or) and `~` (not). Helpers cover the common cases: `isin`, `between` (inclusive on
both ends), `like` (SQL wildcards `%` and `_`), `rlike` (regular expression), `startswith`, `contains`,
`isNull` and `isNotNull`. A SQL string such as `"amount > 100 AND status = 'paid'"` also works.

**Where it matters**
- **Early filtering:** drop rows before joins and aggregations so less data moves.
- **Data quality:** select the rows that break a rule to inspect or quarantine them.

**Watch out for**
- Python's operator precedence: `F.col('a') > 1 & F.col('b') < 2` is wrong. Put each comparison in parentheses.
- Use `&`, `|`, `~`, never Python's `and`, `or`, `not`, which raise an error on columns.
- A comparison with null is never true, so `region != 'North'` drops rows where region is null.

### Example

Filter with combined conditions, `isin`, `between` and a regular expression, then see how nulls behave in a comparison.

In [ ]:
orders.where((F.col("status") == "paid") & (F.col("amount") > 100)).select("order_id", "status", "amount").show()

orders.where(F.col("status").isin("placed", "returned") & F.col("amount").between(50, 100)) \
    .select("order_id", "status", "amount").show()

customers.where(F.col("name").rlike(r"^[A-D]")).select("customer_id", "name").show()

print("region != 'North':", customers.where(F.col("region") != "North").count(), "rows")
print("including nulls:  ", customers.where((F.col("region") != "North") | F.col("region").isNull()).count(), "rows")

**What to notice:** Three paid orders exceed 100. `between(50, 100)` includes both ends. The regular expression kept names starting
with A to D. Customers outside the North number 6, yet `region != 'North'` returned 5: customer 4's null region
made the comparison unknown, so the row was dropped. Add `isNull()` explicitly when nulls should be kept.

## Sort and limit

`orderBy` (or `sort`) sorts by one or more columns; use `F.desc("col")` or `F.col("col").desc()` for descending
order, and `asc_nulls_last()` or `desc_nulls_first()` to place nulls. `limit(n)` keeps the first `n` rows and
returns a DataFrame; `take(n)`, `head(n)` and `first()` return rows to Python.

**Where it matters**
- **Reports and top-N lists:** the five largest orders, newest events first.
- **Deterministic output** for tests and exports.

**Watch out for**
- Without `orderBy`, row order is not guaranteed and can change between runs. Never rely on it.
- A global `orderBy` shuffles all the data into sorted ranges; avoid sorting large tables you will write anyway.
- `limit` without `orderBy` returns some `n` rows, not the first `n` by any column.

### Example

Sort orders by amount, sort customers with nulls last, and take the three newest orders.

In [ ]:
orders.orderBy(F.desc("amount"), "order_id").select("order_id", "amount").show(3)

customers.orderBy(F.col("region").asc_nulls_last(), "customer_id").select("customer_id", "region").show()

newest = orders.orderBy(F.col("order_ts").desc()).limit(3)
print([row.order_id for row in newest.collect()])

**What to notice:** The bulk order 9 comes first by amount. Customer 4, whose region is null, sorts last because of
`asc_nulls_last()`; by default nulls come first in ascending order. The three newest orders are 12, 11 and 10.

## Remove duplicates

`distinct()` keeps one copy of each identical row. `dropDuplicates(["col", ...])` keeps one row per combination
of the listed columns. Both shuffle the data, because identical values must meet in the same partition.

**Where it matters**
- **Exact re-deliveries:** the same file or event sent twice.
- **Lists of values:** the distinct statuses or SKUs in a table.

**Watch out for**
- `dropDuplicates(subset)` keeps an arbitrary row per key, not the latest one. Use a window to choose ([lesson 7](pyspark-07-aggregations-and-windows.ipynb)).
- `count()` after `distinct()` reads all the data; `approx_count_distinct` is cheaper on large tables.

### Example

Find the distinct statuses, and keep one order line per SKU.

In [ ]:
orders.select("status").distinct().orderBy("status").show()
one_per_sku = order_items.dropDuplicates(["sku"])
print("order lines:", order_items.count(), "| distinct SKUs:", one_per_sku.count())

**What to notice:** Four statuses exist. Order lines number 17 but SKUs only 14, because `SKU-4002`, `SKU-4003` and `SKU-4005` each
appear in two orders. Which of the two lines `dropDuplicates` kept for each SKU is not defined.

## Combine rows from two DataFrames

- `union` stacks rows **by column position**; `unionByName` stacks them **by column name**, and with
  `allowMissingColumns=True` fills missing columns with null. Neither removes duplicates; follow with
  `distinct()` for SQL's `UNION`.
- `intersect` keeps rows present in both; `subtract` (SQL `EXCEPT DISTINCT`) keeps rows of the first that are
  not in the second; `exceptAll` and `intersectAll` keep duplicates.

**Where it matters**
- **Appending deliveries** from several files or systems into one table.
- **Reconciliation:** rows in the source but not in the target, and the reverse.

**Watch out for**
- `union` by position silently mixes up columns when two frames list them in different orders. Prefer `unionByName`.
- Set operations compare whole rows, so a timestamp column added to one side makes every row different.

### Example

Two deliveries list the same two columns in a different order. `union` pairs them by position and puts a status
into `order_id`; `unionByName` pairs them by name. Then compare two lists of order ids with set operations.

In [ ]:
monday = spark.createDataFrame([("o101", "paid")], "order_id string, status string")
tuesday = spark.createDataFrame([("shipped", "o102")], "status string, order_id string")

print("union, by position:")
monday.union(tuesday).show()
print("unionByName, by name:")
monday.unionByName(tuesday).show()

source = spark.createDataFrame([(1,), (2,), (3,), (3,)], "order_id int")
target = spark.createDataFrame([(2,), (3,), (4,)], "order_id int")
print("in both:          ", sorted(r.order_id for r in source.intersect(target).collect()))
print("only in source:   ", sorted(r.order_id for r in source.subtract(target).collect()))
print("source exceptAll: ", sorted(r.order_id for r in source.exceptAll(target).collect()))

**What to notice:** `union` raised no error: both columns are strings, so `shipped` landed in `order_id` without complaint. That is
why `unionByName` is the safer default. `subtract` removed every 3 from the source, while `exceptAll` removed
only as many copies as the target has, leaving one 3 behind.

## Sample and split

`sample(fraction, seed)` keeps roughly that fraction of rows, chosen at random; the seed makes it repeatable.
`randomSplit([0.8, 0.2], seed)` splits one DataFrame into several with those proportions, and `sampleBy(col,
fractions, seed)` samples a different fraction from each group (a stratified sample).

**Where it matters**
- **Developing on big data:** iterate on 1 percent of a table, then run on all of it.
- **Machine learning:** train and test splits.

**Watch out for**
- `fraction` is a probability per row, not an exact count. Use `limit` for an exact number of rows.
- Without a seed, every run returns a different sample.

### Example

Sample half of 1,000 rows twice with the same seed, then split them 80/20.

In [ ]:
ids = spark.range(1000)
first = ids.sample(fraction=0.5, seed=7).count()
second = ids.sample(fraction=0.5, seed=7).count()
print("sampled rows:", first, "and", second)

train, test = ids.randomSplit([0.8, 0.2], seed=7)
print("train:", train.count(), "| test:", test.count(), "| total:", train.count() + test.count())

**What to notice:** The same seed returned the same sample both times, close to but not exactly 500 rows. The split covers all
1,000 rows with no overlap, in roughly an 80/20 ratio.

## Bring results back to Python

Actions that return data to the driver: `collect()` (all rows as a list of `Row`), `take(n)` and `head(n)` (the
first `n`), `first()` (one row), and `toLocalIterator()` (rows one partition at a time). A `Row` reads like a
tuple and like an object: `row["sku"]`, `row.sku`, `row.asDict()`. `toPandas()` converts to a pandas DataFrame
and needs pandas installed.

**Where it matters**
- **Small results:** a single total, a list of keys, a lookup dictionary.
- **Notebooks:** a pandas DataFrame for plotting an aggregated result.

**Watch out for**
- `collect()` and `toPandas()` on a large DataFrame move all of it into the driver's memory. Aggregate or `limit` first.
- Only the driver can use the returned Python objects; do not try to use them inside DataFrame expressions.

### Example

Read one aggregated value, turn a small result into a dictionary, and read a `Row` three ways.

In [ ]:
total = orders.agg(F.sum("amount").alias("total")).first()["total"]
print("total revenue:", total)

statuses = {row.status: row["count"] for row in orders.groupBy("status").count().collect()}
print("orders per status:", statuses)

row = order_items.orderBy("order_id", "sku").first()
print(row, "|", row["sku"], "|", row.qty, "|", row.asDict())

**What to notice:** Each result here is tiny: one number, one small dictionary, one row. That is the safe way to use the driver.
The total is a Python `Decimal`, the exact value of the `decimal` column.

## Check yourself

Answer each question in your own words before reading the answer.

**Q1.** Why prefer `unionByName` to `union`?

**A.** `union` matches columns by position, so two frames that list the same columns in a different order are silently mixed up. `unionByName` matches by name.

**Q2.** `customers.where(F.col('region') != 'North')` returned fewer rows than expected. Why?

**A.** A comparison with null is unknown, so rows with a null region are dropped. Add `| F.col('region').isNull()` to keep them.

**Q3.** What is wrong with `F.col('a') > 1 & F.col('b') < 2`?

**A.** Python evaluates `&` before the comparisons. Write `(F.col('a') > 1) & (F.col('b') < 2)`.

**Q4.** Is the output of `limit(5)` the first five rows?

**A.** Only if you sorted first. Without `orderBy`, Spark returns any five rows, and which ones can change.

**Q5.** When is `collect()` safe?

**A.** When the result is small: an aggregate, a sample or a short list. On a large DataFrame it fills the driver's memory.

**Q6.** Which keeps duplicates: `subtract` or `exceptAll`?

**A.** `exceptAll` removes one matching copy per row in the other frame and keeps the rest; `subtract` removes every copy and returns distinct rows.

## Key takeaways

- Create test DataFrames with an explicit schema; inferred types guess.
- Put each comparison in parentheses and combine with `&`, `|`, `~`.
- Nulls fail every comparison: handle them explicitly when filtering.
- Sort before `limit`, `first` or any comparison of output order.
- Use `unionByName`, not `union`.
- Bring only small results to Python.

**Next:** Practise this module in [PySpark drills 2: DataFrame essentials](pyspark-drills-02-dataframe-essentials.ipynb), then continue with [PySpark 3: types, functions and nulls](pyspark-03-types-functions-and-nulls.ipynb).